# Optimización de consultas SQL con Machine Learning — Prototipo en Colab

**Proyecto de titulación · Universidad del Bío-Bío**

Este notebook entrena un modelo que aprende a **elegir el orden de combinación de tablas (JOIN)**
más rápido para una consulta, a partir de su plan de ejecución de PostgreSQL.

Qué hace, de principio a fin:
1. Instala **PostgreSQL dentro de Colab** (no necesitas nada local).
2. Crea un esquema tipo **TPC-H** (región, nación, cliente, proveedor, parte, órdenes, líneas) con **datos sesgados**, para que el orden de JOIN realmente importe.
3. Genera consultas con varios JOIN y, para cada una, **fuerza distintos órdenes** (`join_collapse_limit = 1`).
4. Captura de cada plan las **características** (con `EXPLAIN`) y el **tiempo real** (con `EXPLAIN ANALYZE`, calentamiento + mediana).
5. Entrena **LightGBM** en dos modos: **regresión** (predice el tiempo en ms) y **ranking** (ordena los planes con LambdaMART).
6. Evalúa: MAE/RMSE/R², **acierto del mejor plan** y **% de mejora frente al plan nativo de PostgreSQL**.
7. Muestra la **importancia de características** y **exporta el modelo** (`.pkl`) para el backend.

> Para correrlo: *Entorno de ejecución → Ejecutar todo*. Tarda unos minutos.
> El esquema sintético es un sustituto rápido de TPC-H/JOB; al final se explica cómo cambiarlo por los benchmarks reales.


## 1. Instalación de dependencias y PostgreSQL

In [ ]:
# Instala PostgreSQL y las librerías de ML dentro de Colab
!pip -q install psycopg2-binary lightgbm scikit-learn >/dev/null 2>&1
!apt-get -qq -y install postgresql postgresql-contrib >/dev/null 2>&1
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';" >/dev/null 2>&1
!sudo -u postgres psql -c "DROP DATABASE IF EXISTS optim;" >/dev/null 2>&1
!sudo -u postgres psql -c "CREATE DATABASE optim;" >/dev/null 2>&1
print("PostgreSQL listo.")

## 2. Configuración e importaciones

In [ ]:
import json, random, itertools, io, time, warnings
import numpy as np, pandas as pd
import psycopg2
from statistics import median
warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)

# --- Parámetros que puedes ajustar ---
SCALE        = 0.5     # tamaño de los datos (sube a 1.0 o 2.0 para más; baja para ir más rápido)
WARMUP       = 1       # ejecuciones de calentamiento (no se miden)
REPETICIONES = 3       # mediciones por plan; se toma la mediana
MAX_ORDENES  = 10      # máximo de órdenes de JOIN por consulta
TIMEOUT_MS   = 8000    # descarta un plan si tarda más que esto

CONN = dict(host="localhost", dbname="optim", user="postgres", password="postgres")

def conectar():
    c = psycopg2.connect(**CONN)
    c.autocommit = True          # cada sentencia va en su propia transacción
    return c

conn = conectar()
print("Conectado a:", conn.get_dsn_parameters().get("dbname"))

## 3. Esquema tipo TPC-H y carga de datos (con sesgo)

In [ ]:
def crear_esquema(cur):
    cur.execute('''
    DROP TABLE IF EXISTS lineitem, orders, customer, supplier, part, nation, region CASCADE;
    CREATE TABLE region  (r_regionkey int PRIMARY KEY, r_name text);
    CREATE TABLE nation  (n_nationkey int PRIMARY KEY, n_name text, n_regionkey int);
    CREATE TABLE supplier(s_suppkey int PRIMARY KEY, s_name text, s_nationkey int);
    CREATE TABLE customer(c_custkey int PRIMARY KEY, c_name text, c_nationkey int, c_acctbal numeric);
    CREATE TABLE part    (p_partkey int PRIMARY KEY, p_name text, p_brand text, p_size int);
    CREATE TABLE orders  (o_orderkey int PRIMARY KEY, o_custkey int, o_orderdate date, o_totalprice numeric);
    CREATE TABLE lineitem(l_id int PRIMARY KEY, l_orderkey int, l_partkey int, l_suppkey int,
                          l_quantity int, l_extendedprice numeric);
    ''')

def copy_load(cur, tabla, columnas, filas):
    buf = io.StringIO()
    for f in filas:
        buf.write("\t".join("" if v is None else str(v) for v in f) + "\n")
    buf.seek(0)
    cur.copy_expert(f"COPY {tabla} ({','.join(columnas)}) FROM STDIN", buf)

def generar_datos(cur):
    S = SCALE
    n_sup  = max(50,   int(1000 * S))
    n_cust = max(500,  int(15000 * S))
    n_part = max(500,  int(10000 * S))
    n_ord  = max(2000, int(60000 * S))
    regiones = ["AMERICA","EUROPE","ASIA","AFRICA","MIDDLE EAST"]
    naciones = ["CHILE","ARGENTINA","BRAZIL","PERU","USA","CANADA","GERMANY","FRANCE","SPAIN","ITALY",
                "UK","CHINA","JAPAN","INDIA","KOREA","EGYPT","MOROCCO","KENYA","NIGERIA","SAUDI",
                "IRAN","MEXICO","COLOMBIA","RUSSIA","TURKEY"]
    marcas = [f"Marca#{i:02d}" for i in range(1, 26)]

    copy_load(cur, "region", ["r_regionkey","r_name"], [(i, regiones[i]) for i in range(5)])
    copy_load(cur, "nation", ["n_nationkey","n_name","n_regionkey"],
              [(i, naciones[i], i % 5) for i in range(25)])

    # Sesgo: 60% de clientes y proveedores se concentran en 5 naciones "grandes"
    grandes = [0, 1, 4, 6, 11]
    def nacion_sesgada():
        return random.choice(grandes) if random.random() < 0.6 else random.randrange(25)

    copy_load(cur, "supplier", ["s_suppkey","s_name","s_nationkey"],
              [(i, f"Sup_{i}", nacion_sesgada()) for i in range(n_sup)])
    copy_load(cur, "customer", ["c_custkey","c_name","c_nationkey","c_acctbal"],
              [(i, f"Cli_{i}", nacion_sesgada(), round(random.uniform(-500, 9999), 2)) for i in range(n_cust)])
    copy_load(cur, "part", ["p_partkey","p_name","p_brand","p_size"],
              [(i, f"Parte_{i}", random.choice(marcas), random.randint(1, 50)) for i in range(n_part)])

    base = np.datetime64("2022-01-01")
    ordenes = []
    for i in range(n_ord):
        d = base + int(np.random.randint(0, 1000))
        ordenes.append((i, random.randrange(n_cust), str(d), round(random.uniform(10, 90000), 2)))
    copy_load(cur, "orders", ["o_orderkey","o_custkey","o_orderdate","o_totalprice"], ordenes)

    lid = 0; lote = []
    cols_li = ["l_id","l_orderkey","l_partkey","l_suppkey","l_quantity","l_extendedprice"]
    for o in range(n_ord):
        for _ in range(random.randint(1, 4)):
            lote.append((lid, o, random.randrange(n_part), random.randrange(n_sup),
                         random.randint(1, 50), round(random.uniform(100, 80000), 2)))
            lid += 1
            if len(lote) >= 50000:
                copy_load(cur, "lineitem", cols_li, lote); lote = []
    if lote:
        copy_load(cur, "lineitem", cols_li, lote)
    return lid

def crear_indices(cur):
    for ddl in [
        "CREATE INDEX ON orders(o_custkey)", "CREATE INDEX ON orders(o_orderdate)",
        "CREATE INDEX ON lineitem(l_orderkey)", "CREATE INDEX ON lineitem(l_partkey)",
        "CREATE INDEX ON lineitem(l_suppkey)", "CREATE INDEX ON customer(c_nationkey)",
        "CREATE INDEX ON supplier(s_nationkey)", "CREATE INDEX ON part(p_brand)",
        "CREATE INDEX ON nation(n_regionkey)"]:
        cur.execute(ddl)

t0 = time.time()
with conn.cursor() as cur:
    crear_esquema(cur)
    n_line = generar_datos(cur)
    crear_indices(cur)
    cur.execute("ANALYZE;")
print(f"Datos cargados: lineitem = {n_line:,} filas  ({time.time()-t0:.1f}s)")

## 4. Grafo de JOIN, plantillas de consultas y órdenes forzados

In [ ]:
# Predicados de JOIN entre tablas (el "grafo" del esquema)
EDGES = {
 frozenset({"lineitem","orders"}):   "lineitem.l_orderkey = orders.o_orderkey",
 frozenset({"lineitem","part"}):     "lineitem.l_partkey = part.p_partkey",
 frozenset({"lineitem","supplier"}): "lineitem.l_suppkey = supplier.s_suppkey",
 frozenset({"orders","customer"}):   "orders.o_custkey = customer.c_custkey",
 frozenset({"customer","nation"}):   "customer.c_nationkey = nation.n_nationkey",
 frozenset({"supplier","nation"}):   "supplier.s_nationkey = nation.n_nationkey",
 frozenset({"nation","region"}):     "nation.n_regionkey = region.r_regionkey",
}

def ordenes_conectados(tablas, maximo):
    # Órdenes de JOIN válidos: cada tabla nueva se conecta con alguna ya unida
    res = []
    for perm in itertools.permutations(tablas):
        ok = True
        for i in range(1, len(perm)):
            if not any(frozenset({perm[i], j}) in EDGES for j in perm[:i]):
                ok = False; break
        if ok:
            res.append(perm)
        if len(res) >= maximo * 6:
            break
    random.shuffle(res)
    return res[:maximo]

def construir_sql(orden, filtros):
    sql = f"SELECT count(*) FROM {orden[0]}"
    joined = [orden[0]]
    for t in orden[1:]:
        cond = None
        for j in joined:
            e = frozenset({t, j})
            if e in EDGES:
                cond = EDGES[e]; break
        sql += f" JOIN {t} ON {cond}"
        joined.append(t)
    if filtros:
        sql += " WHERE " + " AND ".join(filtros)
    return sql

def plantillas():
    P = []
    naciones = ["CHILE","USA","GERMANY","CHINA","BRAZIL","FRANCE","JAPAN","SPAIN"]
    regiones = ["AMERICA","EUROPE","ASIA"]
    marcas   = [f"Marca#{i:02d}" for i in [1,5,9,13,17,21]]
    for n in naciones:
        P.append((["lineitem","orders","customer","nation"], [f"nation.n_name = '{n}'"]))
    for m in marcas:
        P.append((["lineitem","part","supplier","nation"], [f"part.p_brand = '{m}'", "part.p_size < 30"]))
    for r in regiones:
        P.append((["lineitem","orders","customer","nation","region"],
                  [f"region.r_name = '{r}'", "orders.o_orderdate >= '2022-06-01'"]))
    for n in naciones[:5]:
        P.append((["lineitem","supplier","nation"], [f"nation.n_name = '{n}'"]))
    return P

CONSULTAS = plantillas()
print("Consultas base:", len(CONSULTAS))

## 5. Captura de planes

Para cada consulta:
- **Plan nativo**: se deja que PostgreSQL elija el orden (línea base a superar).
- **Órdenes forzados**: con `join_collapse_limit = 1` se respeta el orden escrito.
  - `EXPLAIN` (sin ANALYZE) → **características** (estimaciones que sí existen en inferencia).
  - `EXPLAIN ANALYZE` con calentamiento + mediana → **tiempo real** (la etiqueta a predecir).


In [ ]:
def set_sesion(cur, forzar):
    if forzar:
        cur.execute("SET join_collapse_limit=1; SET from_collapse_limit=1; SET geqo=off;")
    else:
        cur.execute("SET join_collapse_limit TO DEFAULT; SET from_collapse_limit TO DEFAULT; SET geqo TO on;")
    cur.execute(f"SET statement_timeout='{TIMEOUT_MS}';")

def plan_estimado(cur, sql):
    cur.execute("EXPLAIN (FORMAT JSON) " + sql)
    return cur.fetchone()[0][0]["Plan"]

def tiempo_real(cur, sql):
    for _ in range(WARMUP):
        cur.execute("EXPLAIN (ANALYZE, FORMAT JSON) " + sql); cur.fetchone()
    ts = []
    for _ in range(REPETICIONES):
        cur.execute("EXPLAIN (ANALYZE, FORMAT JSON) " + sql)
        ts.append(cur.fetchone()[0][0]["Plan"]["Actual Total Time"])
    return median(ts)

filas   = []   # {query_id, orden, plan(estimado), tiempo_ms}
nativos = {}   # query_id -> tiempo real del plan nativo
t0 = time.time()
with conn.cursor() as cur:
    for qid, (tablas, filtros) in enumerate(CONSULTAS):
        # 1) plan nativo (PostgreSQL decide)
        set_sesion(cur, forzar=False)
        try:
            nativos[qid] = tiempo_real(cur, construir_sql(tablas, filtros))
        except Exception:
            continue
        # 2) órdenes forzados
        set_sesion(cur, forzar=True)
        for orden in ordenes_conectados(tablas, MAX_ORDENES):
            sql = construir_sql(orden, filtros)
            try:
                est = plan_estimado(cur, sql)
                tr  = tiempo_real(cur, sql)
            except Exception:
                continue   # p.ej. statement_timeout: se descarta ese plan
            filas.append(dict(query_id=qid, orden=orden, plan=est, tiempo_ms=tr))
print(f"Planes capturados: {len(filas)} | consultas con nativo: {len(nativos)} | {time.time()-t0:.1f}s")

## 6. Ingeniería de características (desde el plan estimado)

In [ ]:
NODOS = ["Seq Scan","Index Scan","Index Only Scan","Bitmap Heap Scan","Bitmap Index Scan",
         "Nested Loop","Hash Join","Merge Join","Hash","Sort","Aggregate","Materialize","Gather"]

def extraer_features(plan, n_tablas):
    cont = {f"n_{nt}": 0 for nt in NODOS}
    agg  = {"n_nodos":0, "prof_max":0, "filas_est_sum":0.0, "costo_sum":0.0, "ancho_sum":0.0, "n_scans":0}
    def rec(n, prof):
        nt = n.get("Node Type", "")
        if nt in NODOS: cont[f"n_{nt}"] += 1
        agg["n_nodos"] += 1
        agg["prof_max"] = max(agg["prof_max"], prof)
        agg["filas_est_sum"] += n.get("Plan Rows", 0) or 0
        agg["costo_sum"]     += n.get("Total Cost", 0) or 0
        agg["ancho_sum"]     += n.get("Plan Width", 0) or 0
        if "Scan" in nt: agg["n_scans"] += 1
        for h in n.get("Plans", []): rec(h, prof + 1)
    rec(plan, 0)
    f = {**cont, **agg}
    f["raiz_costo"]      = plan.get("Total Cost", 0)
    f["raiz_costo_ini"]  = plan.get("Startup Cost", 0)
    f["raiz_filas_est"]  = plan.get("Plan Rows", 0)
    f["raiz_ancho"]      = plan.get("Plan Width", 0)
    f["n_tablas"]        = n_tablas
    return f

registros = []
for row in filas:
    ft = extraer_features(row["plan"], len(row["orden"]))
    ft["query_id"] = row["query_id"]; ft["tiempo_ms"] = row["tiempo_ms"]
    registros.append(ft)

df = pd.DataFrame(registros).fillna(0).sort_values("query_id").reset_index(drop=True)
FEATURES = [c for c in df.columns if c not in ("query_id", "tiempo_ms")]
print(df.shape[0], "planes x", len(FEATURES), "características")
df.head()

## 7. Partición por consulta y entrenamiento (LightGBM)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# La partición se hace POR CONSULTA (no por plan) para evitar fuga de datos.
qids = sorted(df["query_id"].unique())
random.shuffle(qids)
corte   = int(len(qids) * 0.7)
q_train = set(qids[:corte]); q_test = set(qids[corte:])
tr = df[df.query_id.isin(q_train)].copy()
te = df[df.query_id.isin(q_test)].copy()

Xtr, ytr = tr[FEATURES], np.log1p(tr["tiempo_ms"])   # log para estabilizar la escala
Xte, yte = te[FEATURES], np.log1p(te["tiempo_ms"])

# --- Modelo 1: Regresión (predice el tiempo en ms) ---
reg = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31,
                        subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1)
reg.fit(Xtr, ytr)

# --- Modelo 2: Ranking de planes por consulta (LambdaMART) ---
tr_s = tr.sort_values("query_id").copy()
rank = tr_s.groupby("query_id")["tiempo_ms"].rank(method="first", ascending=True)  # 1 = más rápido
size = tr_s.groupby("query_id")["tiempo_ms"].transform("size")
tr_s["rel"] = (size - rank).astype(int)                                            # relevancia: más rápido = mayor
grupos = tr_s.groupby("query_id", sort=True).size().values
rnk = lgb.LGBMRanker(objective="lambdarank", n_estimators=400, learning_rate=0.05,
                     num_leaves=31, random_state=42, verbosity=-1)
rnk.fit(tr_s[FEATURES], tr_s["rel"].values, group=grupos)
print(f"Entrenado con {len(q_train)} consultas ({len(tr)} planes). Prueba: {len(q_test)} consultas.")

## 8. Evaluación (precisión y mejora frente a PostgreSQL)

In [ ]:
pred_te = np.expm1(reg.predict(Xte))
print("== Regresión (conjunto de prueba) ==")
print(f"MAE  = {mean_absolute_error(te['tiempo_ms'], pred_te):.2f} ms")
print(f"RMSE = {mean_squared_error(te['tiempo_ms'], pred_te) ** 0.5:.2f} ms")
print(f"R^2  = {r2_score(np.log1p(te['tiempo_ms']), reg.predict(Xte)):.3f} (en escala log)")

def evaluar(modelo, menor_es_mejor):
    # Por cada consulta de prueba: el modelo elige un plan y se compara con el óptimo y el nativo
    hits = 0; regrets = []; mejoras = []; n = 0
    for qid in sorted(q_test):
        g = te[te.query_id == qid]
        if g.empty or qid not in nativos:
            continue
        s = modelo.predict(g[FEATURES])
        idx = int(np.argmin(np.expm1(s))) if menor_es_mejor else int(np.argmax(s))
        elegido = g.iloc[idx]["tiempo_ms"]
        mejor   = g["tiempo_ms"].min()
        nat     = nativos[qid]
        hits += int(abs(elegido - mejor) < 1e-9)
        regrets.append((elegido - mejor) / max(mejor, 1e-9))
        mejoras.append((nat - elegido) / max(nat, 1e-9))
        n += 1
    return n, hits / n, float(np.median(regrets)), float(np.median(mejoras))

for nombre, mod, menor in [("Regresión", reg, True), ("Ranking", rnk, False)]:
    n, hit, reg_med, mej = evaluar(mod, menor)
    print(f"\n== {nombre} ({n} consultas de prueba) ==")
    print(f"Acierto del mejor plan (top-1): {hit*100:.0f}%")
    print(f"Sobrecosto mediano (regret):    {reg_med*100:.1f}%")
    print(f"Mejora mediana vs PostgreSQL:   {mej*100:+.1f}%")

## 9. Importancia de características

In [ ]:
import matplotlib.pyplot as plt
imp = pd.Series(reg.feature_importances_, index=FEATURES).sort_values().tail(15)
plt.figure(figsize=(7, 5))
imp.plot(kind="barh", color="#2563eb")
plt.title("Importancia de características (modelo de regresión)")
plt.xlabel("Ganancia relativa"); plt.tight_layout(); plt.show()

## 10. Exportar el modelo y el dataset

In [ ]:
import joblib
joblib.dump(reg, "modelo_regresion.pkl")
joblib.dump(rnk, "modelo_ranking.pkl")
with open("features.json", "w") as f:
    json.dump(FEATURES, f, ensure_ascii=False, indent=2)
df.to_csv("dataset_planes.csv", index=False)
print("Guardado: modelo_regresion.pkl, modelo_ranking.pkl, features.json, dataset_planes.csv")

try:
    from google.colab import files
    for fn in ["modelo_regresion.pkl", "modelo_ranking.pkl", "features.json", "dataset_planes.csv"]:
        files.download(fn)
except Exception as e:
    print("Si no se descargan solos, bájalos desde el panel de archivos (carpeta izquierda). Detalle:", e)

## 11. Próximos pasos

- **Cambiar el esquema sintético por los benchmarks reales** de la propuesta (TPC-H con `SF=10` y el Join Order Benchmark sobre IMDb). El resto del notebook (captura de planes, características, entrenamiento y evaluación) queda igual: solo cambia cómo se cargan las tablas y las consultas base.
- **Integrar el modelo al backend** (`analizador/` de la app): cargar `modelo_regresion.pkl` + `features.json`, generar los órdenes candidatos de una consulta nueva, extraer sus características con `EXPLAIN` y recomendar el orden con menor tiempo predicho. Ese resultado alimenta el bloque de recomendación de la pantalla "Optimización de consulta".
- **Subir la escala** (`SCALE`) y el número de plantillas para un modelo más robusto, y guardar el `dataset_planes.csv` para no re-capturar en cada corrida.
